# LionAG2: Recursive Exploratory Research with AG2 beta — simple search agent with exa (1/10)

On May 3rd, 2026. Autogen author and now [AG2](https://www.ag2.ai/) founder and CEO Qingyun Wu co-hosted a hackathon with Fordham Gabelli School of Business, where I graduated from, so I went.

I did a self-explorative research engine that drills deeper into uncertain topics, I named the project **lionag2** using both lionagi and ag2. I am rewriting it to build with only ag2, and will be releasing these as a **10-part tutorial series**.

The following tutorial uses OpenAI and Exa as model and tooling providers, please make sure you have the required API keys, which you can get from:
- https://openai.com/api/
- https://exa.ai/

If you prefer other providers, AG2 also provides a lot of built-in options — check [here](https://docs.ag2.ai/latest/docs/beta/llm-configuration/) for additional model providers, and [here](https://docs.ag2.ai/latest/docs/beta/tools/) for additional search tool providers. The main logic of the code will still work.

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

True

## Your first AG2 agent

An agent is simply an actor interface with runtime configurations, for example, you can declare a simple agent as follows:

In [2]:
from autogen.beta import Agent
from autogen.beta.config import OpenAIConfig
from autogen.beta.events import ToolResultsEvent

config = OpenAIConfig(
    model="gpt-5.4-mini",
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url="https://api.openai.com/v1",
)

agent = Agent("surveyor", config=config)

In AG2 beta, **stream** is a foundational concept — it acts as an event bus, allowing introspection, reactive typed event handling, as well as other features which we will explore in future tutorials. Here we are using it as conversation history.

In [3]:
from autogen.beta import MemoryStream

stream = MemoryStream()
reply = await agent.ask(
    "Concisely, what are the main classes of superconductors?",
    stream=stream,
)

In [4]:
from IPython.display import Markdown

Markdown(reply.body)

The main classes of superconductors are:

- **Conventional superconductors**: Usually **elemental metals and simple alloys** explained well by BCS theory, with **phonon-mediated pairing**.
- **Unconventional superconductors**: Materials where the pairing mechanism is not fully explained by conventional BCS theory, including:
  - **High-\(T_c\) cuprates**
  - **Iron-based superconductors**
  - **Heavy-fermion superconductors**
  - **Organic superconductors**
- **Type I vs. Type II** is another common classification, based on magnetic behavior:
  - **Type I**: Fully expel magnetic fields until a critical field
  - **Type II**: Allow magnetic flux to penetrate in vortices over a mixed state

If you want, I can also give the classification by **pairing mechanism**, **magnetic response**, or **material family** in a single table.

## Search with Exa

AG2 comes with plenty batteries built-in, including common search providers like Exa, DuckDuckGo, Perplexity, and Tavily.

We'll also set up **observers** — typed callbacks that let us observe the agent's actions as they happen. Each observer subscribes to a specific event type (e.g. `ToolCallEvent`, `ToolResultsEvent`), and fires in real-time during `agent.ask()`. This gives us live visibility into every tool call and result without polling or post-hoc inspection.

In [5]:
import json
from autogen.beta.tools import ExaToolkit
from autogen.beta.events import DataInput, ToolCallEvent
from autogen.beta.tools.search.exa import (
    ExaSearchResponse, ExaSearchResult, ExaAnswerResult, ExaContentResult,
)

exa_tool = ExaToolkit(api_key=os.getenv("EXA_API_KEY"))
agent.add_tool(exa_tool)

title_to_url: dict[str, str] = {}
_tool_calls: dict[str, dict] = {}


def _date(d):
    return f" · {d[:10]}" if d else ""


def _hits(results):
    return "\n".join(
        f"  {i}. [{r.title or r.url}]({r.url}){_date(r.published_date)}"
        for i, r in enumerate(results, 1)
    )


def _render(data):
    if isinstance(data, ExaSearchResponse):
        return f"_{len(data.results)} results_\n{_hits(data.results)}"
    if isinstance(data, ExaAnswerResult):
        lines = [data.answer, "", f"_{len(data.citations)} citations_"]
        for c in data.citations:
            lines.append(f"- [{c.title or c.url}]({c.url})")
        return "\n".join(lines)
    if isinstance(data, list) and data:
        if isinstance(data[0], (ExaSearchResult, ExaContentResult)):
            return _hits(data)
    return repr(data)


@agent.observer(ToolCallEvent)
def on_tool_call(event: ToolCallEvent) -> None:
    args = json.loads(event.arguments) if isinstance(event.arguments, str) else event.arguments
    _tool_calls[event.id] = {"name": event.name, "args": args}
    arg_str = ", ".join(f"{k}={v!r}" for k, v in args.items())
    display(Markdown(f"**`{event.name}`**({arg_str})"))


@agent.observer(ToolResultsEvent)
def on_tool_results(event: ToolResultsEvent) -> None:
    for r in event.results:
        for p in r.result.parts:
            if isinstance(p, DataInput) and isinstance(p.data, ExaSearchResponse):
                for hit in p.data.results or []:
                    if hit.title and hit.url:
                        title_to_url[hit.title] = hit.url
        data = r.result.parts[0].data
        display(Markdown(_render(data)))

A `Toolkit` in AG2 is composed of multiple tools — in this case, the Exa toolkit contains `search`, `find_similar`, `get_contents`, and `answer`. You can check the API for these [here](https://docs.exa.ai/). When a toolkit is provided to an agent, by default it enables access to all the tools it carries.

We attached two **observers** — typed callbacks that fire as the agent works:
- `on_tool_call` displays each tool invocation as it happens
- `on_tool_results` renders search results and captures URLs into `title_to_url`

Observers can be sync or async. When the next cell runs, watch each step stream out live.

Since we re-use the same stream, the agent resumes the conversation with previous context intact.

In [6]:
reply = await agent.ask(
    "With exa search, concisely introduce the frontier of each class",
    stream=stream,
)

Markdown(reply.body)

**`exa_answer`**(query='Frontier of type I and type II superconductors recent advances vortex matter materials 2024 overview citations')

**`exa_answer`**(query='Frontier of unconventional superconductors cuprates iron-based heavy fermion organic recent advances 2024 overview citations')

**`exa_answer`**(query='Frontier of conventional superconductors recent advances highest Tc ambient pressure hydrides overview 2024 citations')

Recent advances in the field of conventional superconductors, particularly hydrides, have significantly pushed the boundaries of achievable critical temperatures (Tc) at ambient pressure. Historically, high-temperature superconductivity in hydrides was primarily observed under high-pressure conditions, often exceeding hundreds of gigapascals, with record Tc values reaching up to 260 K in compounds like LaH10 ([Sun, Ying, 2024](https://ui.adsabs.harvard.edu/abs/2024NSRev..11D.270S/abstract)). However, recent research has focused on discovering hydrides that exhibit high Tc at or near ambient pressure, which is crucial for practical applications.

One promising avenue involves the study of few-hydrogen metal-bonded hydrides, which have shown potential as ductile, high-Tc superconductors under ambient conditions ([Jun-jie Shi et al., 2024](https://iopscience.iop.org/article/10.1088/1361-648X/ad68b3)). Additionally, the exploration of clathrate metal superhydrides, such as LaH10, has provided insights into structures that could stabilize high Tc at lower pressures, with some recent theoretical and experimental work suggesting pathways to ambient-pressure superconductivity ([Sun, Ying, 2024](https://ui.adsabs.harvard.edu/abs/2024NSRev..11D.270S/abstract)). Notably, the maximum Tc for some conventional superconductors at ambient pressure has been reported to approach or surpass 20 K, with ongoing research aiming to elevate this further ([Nature Communications, 2025](http://www.nature.com/articles/s41467-025-63702-w)).

Overall, the frontier of this research involves understanding and engineering novel hydride structures that can sustain high Tc without extreme pressures, which could revolutionize superconductor technology in the near future. The field continues to evolve rapidly, with recent breakthroughs indicating that ambient-pressure high-Tc hydrides are within reach, driven by advances in material synthesis, theoretical modeling, and high-pressure experimentation ([Mao Ho-kwang, 2024](https://sharps.ac.cn/uploads/soft/20240914/1-240914121F3521.pdf)).

_8 citations_
- [A new perspective on ductile high-Tc superconductors under ambient pressure: few-hydrogen metal-bonded hydrides - IOPscience](https://iopscience.iop.org/article/10.1088/1361-648X/ad68b3)
- [Clathrate metal superhydrides under high-pressure conditions: enroute to room-temperature superconductivity - ADS](https://ui.adsabs.harvard.edu/abs/2024NSRev..11D.270S/abstract)
- [The maximum Tc of conventional superconductors at ambient pressure | Nature Communications](http://www.nature.com/articles/s41467-025-63702-w)
- [Pressure-induced hydrogen-dominant high-temperature superconductors](https://sharps.ac.cn/uploads/soft/20240914/1-240914121F3521.pdf)
- [A perspective on reducing stabilizing pressure for high-temperature superconductivity in hydrides - IOPscience](https://iopscience.iop.org/article/10.1088/1361-648X/ad7217/meta)
- [Compressed superhydrides: the road to room temperature superconductivity - IOPscience](https://iopscience.iop.org/article/10.1088/1361-648X/ac4eaf)
- [Feasible Route to High-Temperature Ambient-Pressure Hydride Superconductivity](https://journals.aps.org/prl/abstract/10.1103/PhysRevLett.132.166001)
- [[2408.07477] Ternary superhydrides under pressure of Anderson's theorem: Near-record superconductivity in (La,Sc)H$_{12}$](https://arxiv.org/abs/2408.07477)

Recent advances in the field of unconventional superconductors have significantly expanded our understanding of their complex behaviors and underlying mechanisms. In 2024, notable progress includes comprehensive reviews of high-temperature superconductors such as cuprates, iron-based compounds, and nickelates. These studies highlight experimental breakthroughs, such as achieving superconductivity up to 96K in nickelates under pressure, and emphasize ongoing efforts to attain room-temperature superconductivity through pressure and chemical modifications ([zenodo.org](https://zenodo.org/records/18708685)). Theoretical research has also advanced, exploring the role of strong electron correlations, symmetry-breaking phenomena like nematic and smectic orders, and exotic density waves driven by quantum interference effects in Fe-based and cuprate systems ([arXiv:2209.00539](https://arxiv.org/abs/2209.00539)).

Further, the recent literature discusses the persistent mysteries surrounding unconventional superconductivity, such as the case of Sr₂RuO₄, whose pairing mechanism remains elusive despite decades of study ([arXiv:2402.12117](https://arxiv.org/abs/2402.12117)). Advances also extend into the realm of twisted superconductors, where moiré patterns and layer interactions lead to novel phenomena, including topological states and symmetry breaking, opening new avenues for research in moiré and layered materials ([arXiv:2503.23683](https://arxiv.org/pdf/2503.23683)). Additionally, studies on quantum critical metals reveal how loss of quasiparticles near quantum critical points influences superconductivity, highlighting the importance of Kondo destruction and strange-metal behavior in heavy-fermion systems ([Nature Physics, 2024](https://nature.com/nphys)). Overall, these recent developments underscore a vibrant and rapidly evolving frontier in unconventional superconductivity, combining experimental breakthroughs with sophisticated theoretical insights.

_5 citations_
- [High-Temperature Superconductors: Recent Advances in Cuprate, Iron-Based, and Nickelate Systems 2026](https://zenodo.org/records/18708685)
- [[2402.12117] Still mystery after all these years -- Unconventional superconductivity of Sr2RuO4 --](https://arxiv.org/abs/2402.12117)
- [https://arxiv.org/pdf/2503.23683](https://arxiv.org/pdf/2503.23683)
- [[2209.00539] Contents](https://ar5iv.labs.arxiv.org/html/2209.00539)
- [Quantum critical metals and loss of quasiparticles | Nature Physics](http://www.nature.com/articles/s41567-024-02679-7)

Recent advances in the field of superconductivity, particularly concerning vortex matter in Type I and Type II superconductors, are well-documented in 2024 research articles. In Type II superconductors, significant progress has been made in understanding vortex dynamics, pinning mechanisms, and vortex lattice structures. For example, studies have employed microscopic and self-consistent models such as the Bogoliubov–de Gennes framework to explore vortex clustering and stripe formations, which are characteristic of complex vortex interactions in multiband and unconventional superconductors ([arXiv:2404.11491](https://arxiv.org/pdf/2404.11491)). Additionally, recent research highlights the influence of disorder and multiband effects on the intertype domain, showing that disorder can significantly expand the range of parameters where mixed or intermediate vortex states occur ([Springer Nature, 2024](https://link.springer.com/article/10.1007/s11467-023-1379-y)). 

In the realm of vortex pinning and vortex matter materials, recent studies have focused on microscopic mechanisms of vortex pinning, which are crucial for enhancing superconductor performance in applications. These investigations include the effects of defects, pinning landscapes, and vortex interactions under varying magnetic fields, providing insights into optimizing superconductor properties ([PhysRevX, 2024](https://journals.aps.org/prx/abstract/10.1103/PhysRevX.14.041039)). Overall, the recent advances emphasize a multidisciplinary approach combining experimental, theoretical, and computational techniques to deepen understanding of vortex matter, with implications for both fundamental physics and technological applications.

_8 citations_
- [https://arxiv.org/pdf/2404.11491](https://arxiv.org/pdf/2404.11491)
- [Intertype superconductivity evoked by the interplay of disorder and multiple bands | Frontiers of Physics | Springer Nature Link](https://link.springer.com/article/10.1007/s11467-023-1379-y)
- [Revealing the Microscopic Mechanism of Elementary Vortex Pinning in Superconductors](https://journals.aps.org/prx/abstract/10.1103/PhysRevX.14.041039)
- [Condensed Matter > Superconductivity](https://arxiv.org/abs/2407.08547)
- [Condensed Matter > Superconductivity](https://arxiv.org/abs/2411.05551)
- [[2408.06209] Observation of vortex stripes in UTe$_2$](https://arxiv.org/abs/2408.06209)
- [Case studies on time-dependent Ginzburg-Landau simulations for superconducting applications](https://arxiv.org/html/2403.03729v1)
- [[2105.00055] Perspective: Challenges and Transformative Opportunities in Superconductor Vortex Physics](https://arxiv.org/abs/2105.00055)

Here’s a concise “frontier” view of each main class:

- **Conventional superconductors**: frontier is **hydrides** and other phonon-mediated systems, where the goal is **high \(T_c\)** with **lower pressure or ambient pressure**. The current push is toward materials like clathrate superhydrides and related ambient-pressure candidates.

- **Unconventional superconductors**: frontier is **understanding the pairing mechanism** in strongly correlated systems—especially **cuprates, iron-based superconductors, heavy fermions, nickelates, and twisted/moiré systems**. The key challenge is linking complex normal-state physics to the superconducting order.

- **Type I vs. Type II superconductors**: frontier is really in **Type II vortex matter**—how vortices **nucleate, pin, move, cluster, and dissipate**, since this controls critical currents and applications. A newer edge is **intertype / type-1.5 behavior** in multiband systems, where vortex phases become richer than the simple type-I/type-II split.

If you want, I can compress this into a **3-line table**.

## Captured URLs

`on_tool_results` captured every search hit into `title_to_url` while the agent was working — no post-processing needed.

In [7]:
lines = [f"**{len(title_to_url)} URLs captured by observer:**", ""]
for title, url in title_to_url.items():
    lines.append(f"- [{title}]({url})")
display(Markdown(lines and "\n".join(lines) or "No URLs captured (agent may not have searched)."))

**0 URLs captured by observer:**


## Introspection

The observers rendered each step live. You can also walk the stream programmatically — every event is typed and accessible.

In [8]:
from autogen.beta.events import ModelRequest, ModelResponse

events = await stream.history.get_events()
print(f"{len(events)} events in stream:\n")

for ev in events:
    if isinstance(ev, ModelRequest):
        print(f"  >> {ev.parts[0].content[:80]}")
    elif isinstance(ev, ToolCallEvent):
        print(f"     call: {ev.name}()")
    elif isinstance(ev, ToolResultsEvent):
        print(f"     results: {len(ev.results)} part(s)")
    elif isinstance(ev, ModelResponse):
        body = ev.content or ""
        print(f"  << {body[:80]}{'…' if len(body) > 80 else ''}")

13 events in stream:

  >> Concisely, what are the main classes of superconductors?
  << The main classes of superconductors are:

- **Conventional superconductors**: Us…
  >> With exa search, concisely introduce the frontier of each class
  << 
     call: exa_answer()
     call: exa_answer()
     call: exa_answer()
     results: 3 part(s)
  << Here’s a concise “frontier” view of each main class:

- **Conventional supercond…


## Up next

Day 2: **structured outputs**.

Right now `reply.body` is a string — but what if you want the agent to return a typed Pydantic object with a topic, summary, list of open questions, and a novelty score, validated automatically?

That's where `response_schema` comes in, and it enables programmatic operations of agentic processes.